# Greek Banking Sector — ETL: Annual Report PDFs → SQLite

This notebook runs the data pipeline end to end and shows what it does.

**Entities (the companies listed on the Athens Exchange in 2022–2024)**
| Bank | Listed group | Reports used |
|------|--------------|--------------|
| Alpha Bank | Alpha Services and Holdings S.A. | FY2022, FY2023, FY2024 |
| Eurobank | Eurobank Ergasias Services and Holdings S.A. | FY2022, FY2023, FY2024 |
| NBG | National Bank of Greece S.A. | FY2022, FY2023, FY2024 |
| Piraeus Bank | Piraeus Financial Holdings S.A. | FY2022, FY2023, FY2024 |

**Basis:** each year comes from the latest report that contains it, so every year-on-year change compares figures prepared the same way: 2024 and 2023 from the FY2024 report (2023 restated), 2022 from the FY2023 report (2022 restated). Regulatory ratios (CET1, total capital, NPE, DTC, RWAs) are taken as reported at each year end, from the report of that year.

**Steps**
1. `pipeline/extract_statements.py` reads the consolidated income statement and balance sheet pages with `pdfplumber` and parses each line item (8 PDFs, 16 pages).
2. `data/manual/regulatory_ratios.csv` holds the ratios, which appear in the text of the reports, with the PDF page of each one.
3. `pipeline/build_dataset.py` derives the KPIs and writes the processed CSVs and a source map.
4. `rebuild_db.py` loads everything into SQLite.

The PDFs are not in the repo (size). The download links are in `pipeline/sources.py`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

PIPE = Path('..') / 'pipeline'
sys.path.insert(0, str(PIPE.resolve()))
sys.path.insert(0, str(Path('..').resolve()))

from sources import REPORTS, RAW_DIR, STATEMENT_PAGES, ENTITIES

status = pd.DataFrame([
    {'bank': bank, 'year': year, 'entity': ENTITIES[bank], 'file': rep['file'],
     'present': (RAW_DIR / rep['file']).exists()}
    for (bank, year), rep in REPORTS.items()
])
print(status.to_string(index=False))

        bank  year                                               entity                       file  present
  Alpha Bank  2022             Alpha Services and Holdings S.A. (group)    alpha_holdings_2022.pdf    False
  Alpha Bank  2023             Alpha Services and Holdings S.A. (group)    alpha_holdings_2023.pdf     True
  Alpha Bank  2024             Alpha Services and Holdings S.A. (group)    alpha_holdings_2024.pdf     True
    Eurobank  2022 Eurobank Ergasias Services and Holdings S.A. (group) eurobank_holdings_2022.pdf     True
    Eurobank  2023 Eurobank Ergasias Services and Holdings S.A. (group) eurobank_holdings_2023.pdf     True
    Eurobank  2024 Eurobank Ergasias Services and Holdings S.A. (group) eurobank_holdings_2024.pdf     True
         NBG  2022                 National Bank of Greece S.A. (group)               nbg_2022.pdf     True
         NBG  2023                 National Bank of Greece S.A. (group)               nbg_2023.pdf     True
         NBG  2024          

In [2]:
# Example: the lines the parser reads on one page (Alpha Services and Holdings, FY2024 income statement)
import pdfplumber
page = STATEMENT_PAGES['Alpha Bank'][2024]['IS']
with pdfplumber.open(RAW_DIR / REPORTS[('Alpha Bank', 2024)]['file']) as pdf:
    text = pdf.pages[page - 1].extract_text()
print(f'alpha_holdings_2024.pdf, page {page}:\n')
print('\n'.join(text.split('\n')[:26]))

alpha_holdings_2024.pdf, page 246:

CONSOLIDATED FINANCIAL STATEMENTS AS AT 31.12.2024
Consolidated Income Statement
(Amounts in millions of Euro) From 1 January to
Note 31.12.2024 31.12.2023 as restated
Interest and similar income 4,408 3,582
Interest expense and similar charges (2,763) (1,923)
Net interest income 3 1,645 1,659
- of which: net interest income based on the effective interest rate 1,730 1,734
Fee and commission income 480 433
Commission expense (60) (59)
Net fee and commission income 4 420 374
Dividend income 5 6 4
Gains less losses on derecognition of financial assets measured at amortised cost 6 31 (17)
Gains less losses on financial transactions 7 80 70
Other income 8 40 39
Total income from banking operations 2,222 2,129
Staff costs 9 (370) (333)
General administrative expenses 10 (317) (325)
Depreciation and amortization 26,27,28 (179) (157)
Total expenses (866) (815)
Impairment losses and provisions to cover credit risk 11 (360) (381)
Expenses relating to credit r

In [3]:
# Step 1: extract every line item used, with its source page
import extract_statements
lines = extract_statements.extract()
lines.to_csv(extract_statements.OUT, index=False)
print(f'{len(lines)} line items extracted')
lines.pivot_table(index=['bank', 'metric'], columns='year', values='value')

232 line items extracted


year                                                  2022     2023     2024
bank         metric                                                         
Alpha Bank   Additional Tier 1 instruments             0.0    400.0    700.0
             Cash and balances with central banks  12894.8   4219.0   2998.0
             Deposits from customers               50760.9  48449.0  51032.0
             Due from credit institutions           1368.1   1572.0   2296.0
             Due to credit institutions            14344.9   6921.0   6533.0
...                                                    ...      ...      ...
Piraeus Bank Operating income                       2582.0   2607.0   2757.0
             Profit before tax                      1037.0   1078.0   1436.0
             Total assets                          74645.0  76450.0  80044.0
             Total equity                           6581.0   7353.0   8273.0
             Total liabilities                     68064.0  69097.0  71771.0

[78 rows x 3 columns]

In [4]:
# Step 2-3: build the processed tables and KPIs
import build_dataset
kpis = build_dataset.build()
kpis[['bank', 'year', 'nii', 'net_profit', 'total_assets', 'roe', 'nim', 'cost_to_income', 'cet1', 'npe_ratio']]

income_statement_final.csv  96 rows
balance_sheet_final.csv     159 rows
kpis_final.csv              12 rows x 39 columns
sources.csv                 335 rows


,bank,year,nii,net_profit,total_assets,roe,nim,cost_to_income,cet1,npe_ratio
0,Alpha Bank,2022,1173.8,368.1,78011.4,6.0,1.55,48.9,13.20,7.8
1,Alpha Bank,2023,1659.0,618.0,72421.0,9.1,2.21,38.3,14.40,6.0
2,Alpha Bank,2024,1645.0,654.0,72075.0,8.5,2.28,39.0,16.30,3.8
3,Eurobank,2022,1480.0,1347.0,81474.0,22.1,1.86,28.2,16.00,5.2
4,Eurobank,2023,2174.0,1140.0,79781.0,15.7,2.70,31.4,16.90,3.5
5,Eurobank,2024,2507.0,1448.0,101150.0,17.2,2.77,32.9,16.80,2.9
6,NBG,2022,1369.0,1120.0,78113.0,18.4,1.69,36.3,16.60,5.2
7,NBG,2023,2263.0,1106.0,74584.0,15.7,2.96,33.6,17.80,3.7
8,NBG,2024,2356.0,1158.0,74957.0,14.4,3.15,36.5,18.30,2.6
9,Piraeus Bank,2022,1353.0,950.0,74645.0,15.4,1.75,34.4,13.04,6.8


In [5]:
# Step 4: load into SQLite
import rebuild_db
rebuild_db.rebuild()

  kpis                 12 rows
  income_statement     96 rows
  balance_sheet       159 rows
  sources             335 rows


  eba_stress_test       4 rows
  ecb_funding          12 rows
  Sector NII 2024 (€m)                 = 8596.0
  Alpha Bank net profit 2024 (€m)      = 654.0
  Alpha Bank total assets 2024 (€m)    = 72075.0
  NBG CET1 2024 (%)                    = 18.3
  Numbers without a source             = 0

Database rebuilt: 02_Banking_Sector_Dashboard/data/greek_banking_final.db
Copied to:        docs/data/greek_banking_final.db


In [6]:
# Checks: balance sheet identity and source coverage
bal = pd.read_csv('../data/processed/balance_sheet_final.csv')
w = bal.pivot_table(index=['bank', 'year'], columns='metric', values='value')
gap = (w['Total assets'] - w['Total liabilities'] - w['Total equity']).abs()
assert (gap <= 0.2).all(), gap

src = pd.read_csv('../data/processed/sources.csv')
assert src['source'].notna().all()

print(f'Balance sheet identity holds for all 12 bank-years (max gap €{gap.max():.1f}m).')
print(f'{len(src)} numbers in the dataset, each with a report and page reference.')
src.sample(8, random_state=1)

Balance sheet identity holds for all 12 bank-years (max gap €0.0m).
335 numbers in the dataset, each with a report and page reference.


,bank,year,metric,value,source,how
80,Alpha Bank,2024,npe_ratio,3.8,alpha_holdings_2024.pdf p.21,Management report
59,Alpha Bank,2024,Deposits from customers,51032.0,alpha_holdings_2024.pdf p.248,As printed in the statement
258,Piraeus Bank,2022,Cash and balances with central banks,9653.0,pfh_2023.pdf p.174,As printed in the statement
284,Piraeus Bank,2023,Cash and balances with central banks,10567.0,pfh_2024.pdf p.425,As printed in the statement
191,NBG,2022,Profit from operations before impairments,1500.0,nbg_2023.pdf p.242,Operating income + operating expenses
90,Eurobank,2022,Deposits from customers,57239.0,eurobank_holdings_2023.pdf p.146,As printed in the statement
11,Alpha Bank,2022,Goodwill and intangible assets,474.7,alpha_holdings_2023.pdf p.152,As printed in the statement
111,Eurobank,2022,dtc,3402.0,eurobank_holdings_2022.pdf p.6,Eligible DTAs for conversion to tax credits
